# Version 4
New features:
- Flexible architecture
- Hyperparameter tuning tools via Optuna
- Progress bar
- Refactored train_epoch()
- Removed some nice features (such as plotting, i.e., making use of saving metrics) for using a single model.

## Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.mticker as mticker
import optuna
from utils.helper_utils import NestedProgressBar

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Helper functions

In [ ]:
def plot_training_metrics(metrics):
    """
    Plots the training and validation metrics from a model training process.

    This function generates two side-by-side plots:
    1. Training Loss vs. Validation Loss.
    2. Validation Accuracy.

    Args:
        metrics (list): A list or tuple containing three lists:
                        [train_losses, val_losses, val_accuracies].
    """
    # Unpack the metrics into their respective lists
    train_losses, val_losses, val_accuracies = metrics
    
    # Determine the number of epochs from the length of the training losses list
    num_epochs = len(train_losses)
    # Create a 1-indexed range of epoch numbers for the x-axis
    epochs = range(1, num_epochs + 1)

    # Create a figure and a set of subplots with 1 row and 2 columns
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # --- Configure the first subplot for training and validation loss ---
    # Select the first subplot
    ax1 = axes[0]
    # Plot training loss data
    ax1.plot(epochs, train_losses, color='#085c75', linewidth=2.5, marker='o', markersize=5, label='Training Loss')
    # Plot validation loss data
    ax1.plot(epochs, val_losses, color='#fa5f64', linewidth=2.5, marker='o', markersize=5, label='Validation Loss')
    # Set the title and axis labels for the loss plot
    ax1.set_title('Training & Validation Loss', fontsize=14)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    # Display the legend
    ax1.legend()
    # Add a grid for better readability
    ax1.grid(True, linestyle='--', alpha=0.6)

    # --- Configure the second subplot for validation accuracy ---
    # Select the second subplot
    ax2 = axes[1]
    # Plot validation accuracy data
    ax2.plot(epochs, val_accuracies, color='#fa5f64', linewidth=2.5, marker='o', markersize=5, label='Validation Accuracy')
    # Set the title and axis labels for the accuracy plot
    ax2.set_title('Validation Accuracy', fontsize=14)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    # Display the legend
    ax2.legend()
    # Add a grid for better readability
    ax2.grid(True, linestyle='--', alpha=0.6)
    
    # --- Apply dynamic and consistent styling to both subplots ---
    # Calculate a suitable interval for the x-axis ticks to avoid clutter
    x_interval = (num_epochs - 1) // 10 + 1

    # Loop through each subplot to apply common axis settings
    for ax in axes:
        # Set the y-axis to start at 0 and the x-axis to span the epochs
        ax.set_ylim(bottom=0)
        ax.set_xlim(left=1, right=num_epochs)
        
        # Set the major tick locator for the x-axis using the dynamic interval
        ax.xaxis.set_major_locator(mticker.MultipleLocator(x_interval))
        # Set the font size for the tick labels on both axes
        ax.tick_params(axis='both', which='major', labelsize=10)

    # Adjust subplot parameters for a tight layout
    plt.tight_layout()
    # Display the plots
    plt.show()

## Data prepping

In [ ]:
# Define data paths
data_paths = ['./data']  # TODO: Update with actual data paths

In [ ]:
# Create datasets (if applicable)
## Inspect data
# Get sample and label from 0th index
# Print shapes and types
# Print labels and PV locations

In [ ]:
# Visualize sample and corresponding label (create in utils)

## Create the model
This will be the simplest form of the model possible, no bells and whistles, 
completely hardcoded.

In [ ]:
class CNNBlock(nn.Module):
    """
    Defines a single convolutional block for a CNN.

    This block consists of a convolutional layer, batch normalization,
    a ReLU activation, and a max-pooling layer, bundled as a sequential module.
    """
    def __init__(self, in_channels, out_channels, kernel_size, padding):
        """
        Initializes the layers of the CNNBlock.

        Args:
            in_channels (int): Number of channels in the input image.
            out_channels (int): Number of channels produced by the convolution.
            kernel_size (int, optional): Size of the convolving kernel. Defaults to 3.
            padding (int, optional): Zero-padding added to both sides of the input. Defaults to 1.
        """
        super(CNNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, 
                      padding=padding),
            nn.BatchNorm1d(out_channels),
            nn.LeakyReLU()  # Could start with ReLU instead and switch to LeakyReLU later
            # Can't do MaxPool1d because we need equal sized input and output
            # nn.MaxPool1d(kernel_size=2, stride=2)
        )
    
    def forward(self, x):
        """
        Defines the forward pass of the CNNBlock.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_channels, length).

        Returns:
            torch.Tensor: Output tensor after applying the convolutional block.
        """
        return self.block(x)

In [ ]:
# Verify the CNNBlock
print("--- Verifying CNNBlock ---\n")

# Instantiate the block with 1 input channel and 1 output channel
verify_cnn_block = CNNBlock(in_channels=1, out_channels=1)
print("Block Structure:\n")
print(verify_cnn_block)

# Verify the output shape after a forward pass
# Create a dummy input tensor (batch_size=1, channels=1, length=4000), i.e.,
# a single 4000 bin histogram
dummy_input = torch.randn(1, 1, 4000)
print(f"\nInput tensor shape:  {dummy_input.shape}")

# Pass the dummy tensor through the block
output = verify_cnn_block(dummy_input)
print(f"Output tensor shape: {output.shape}")

In [ ]:
class FlexibleCNN(nn.Module):
    """
    A flexible Convolutional Neural Network (CNN) model for 1D data.

    This model allows customization of the number of convolutional layers,
    channels, and kernel sizes.
    """
    def __init__(self, in_channels, n_layers, n_filters, kernel_sizes, 
                 dropout_rate):
        """
        Initializes the FlexibleCNN model.

        Args:
            in_channels (int): Number of input channels. Defaults to 1 (single
                               KDE), but can be more for KDE-A, KDE-B, xMax,
                               and yMax.
            n_layers: The number of convolutional blocks to create.
            n_filters: A list of integers specifying the number of output
                       filters for each convolutional block.
            kernel_sizes: A list of integers specifying the kernel size for
                          each convolutional layer.
            dropout_rate: The dropout probability to be used in the classifier.
            fc_size: The number of neurons in the hidden fully connected layer.
        """
        super(FlexibleCNN, self).__init__()
        
        # Initialize empty list to hold convolutional blocks
        blocks = []

        for i in range(n_layers):

            # Get parameters for currently convolutional block
            out_channels = n_filters[i]
            kernel_size = kernel_sizes[i]
            padding = (kernel_size - 1) // 2  # same padding

            # Define CNNBlock to add to model
            block = CNNBlock(in_channels, out_channels, kernel_size, padding)
            
            # Add block to the list
            blocks.append(block)

            # Update number of input channels for next block
            in_channels = out_channels

        # Combine blocks into single sequential module
        self.features = nn.Sequential(*blocks)

        # Final 1x1 convolution to reduce to single channel output for 
        # classification
        self.head = nn.Sequential(
            nn.Conv1d(in_channels, 1, kernel_size=1),
        )

    def forward(self, x):
        """
        Defines the forward pass of the FlexibleCNN model.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_channels, length).

        Returns:
            torch.Tensor: Output tensor with class scores.
        """
        # Pass input through convolutional blocks to extract features
        x = self.features(x)

        # Pass features through locator to get per-bin output
        x = self.head(x)  # (batch_size, 1, length)
        return x.squeeze(1)   # (batch_size, length)

In [ ]:
# Verify the SimpleCNN
print("--- Verifying FlexibleCNN ---\n")

# Verify the structure of the model
# Instantiate the model with 15 output classes
verify_flexible_cnn = FlexibleCNN(in_channels=1, 
                                  n_layers=3, 
                                  n_filters=[16, 32, 64], 
                                  kernel_sizes=[3, 3, 3], 
                                  dropout_rate=0.5)
print("Model Structure:\n")
print(verify_flexible_cnn)

# Verify the output shape after a forward pass
# Create a dummy input tensor (batch_size=1, channels=1, length=4000)
dummy_input = torch.randn(1, 1, 4000)
print(f"\nInput tensor shape:  {dummy_input.shape}")

# Pass the dummy tensor through the model
output = verify_flexible_cnn(dummy_input)
print(f"Output tensor shape: {output.shape}")

## Define training and evaluation steps

In [ ]:
# TODO: untangle mess model loops scattered all throughout pv-finder, in loss.py
# / alt_loss_A.py, training.py, utilities.py (progress bar, etc.), device
# allocation, epoch loop, train loop, validation loop
#
# I will sketch out the training loop below, but it will need to be refactored
# for pv-finder's specific needs and to ensure clean separation of concerns.

In [ ]:
# Define training loop for one epoch (one pass through the training data)
def train_epoch(model, train_loader, loss_function, optimizer, device, pbar):
    model.to(device)  # Move model to the appropriate device
    model.train()  # Set model to training mode
    total_loss = 0
    n_correct_predictions, n_total_predictions = 0, 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Update progress bar for batch
        pbar.update_batch(batch_idx + 1)

        # Move data to the appropriate device
        inputs, targets = inputs.to(device), targets.to(device)
        # Zero the parameter gradients
        optimizer.zero_grad()
        # Forward pass
        outputs = model(inputs)
        # Compute loss
        loss = loss_function(outputs, targets)
        # Backpropagation to compute gradients
        loss.backward()
        # Update parameters
        optimizer.step()

        # Accumulate total loss
        # Multiply by batch size to get total loss for the batch
        total_loss += loss.item() * inputs.size(0) 

        # Update total sample count
        n_total += len(targets)  # Might be targets.size(0) if targets is a tensor

        # TODO: Get predicted PV locations
        # TODO: Understand Simon's implementation
        # _, predicted = outputs.max(1) --- IGNORE ---

        # TODO: Update total correct predictions
        # n_correct += predicted.eq(targets).sum().item() --- IGNORE ---
        n_correct = None

    # Calculate average loss for epoch
    epoch_loss = total_loss / n_total
    # Calculate average accuracy for the epoch
    epoch_accuracy = n_correct / n_total
    print(f"Epoch loss (avg.): {epoch_loss:.3f} | Accuracy: {epoch_accuracy*100:.2f}%")

    return model, total_loss

In [ ]:
# NOTE: Not used.
def train_model(model, train_dataloader, loss_function, optimizer, device, n_epochs):
    """
    Trains model for specified number of epochs.

    Manages the overall training loop, but does NOT perform validation. It
    initializes a progress bar to visualize progress and, for each epoch, calls
    a helper function to perform the training steps. Handles periodic logging of
    the training loss, and it provides a layer that will eventually allow for
    integrating further features like mlflow logging, checkpointing, etc.

    Args:
        model: The neural network model to be trained.
        train_dataloader: The DataLoader providing the training data.
        loss_function: The loss function used for training.
        optimizer: The optimization algorithm (e.g., Adam, SGD).
        device: The device ('cpu' or 'cuda') to perform training on.
        n_epochs: The total number of epochs to train the model.
    """

    # Initialize progress bar for training
    pbar = NestedProgressBar(
        total_epochs=n_epochs,
        total_batches=len(train_dataloader),
        epoch_message_frequency=5,  # Placeholder value
        mode='train',
    )

    for epoch in range(n_epochs):
        # Update epoch-level progress bar
        pbar.update_epoch(epoch + 1)

        # Call helper function to train model for one epoch
        _, train_loss = train_epoch(model,
                                    train_dataloader, 
                                    loss_function, 
                                    optimizer, 
                                    device, 
                                    pbar)

        # Log training loss for current epoch at specified intervals
        pbar.maybe_log_epoch(
            epoch + 1,
            message=f"Epoch {epoch+1} - Train loss: {train_loss:.4f}"
        )
    
    # Close progress bar and print final message upon completion
    pbar.close(message="Training complete.\n")

In [ ]:
def validate(model, val_loader, loss_function, device):
    """
    Finds the validation loss and accuracy of a model on a given dataset.

    This function iterates through validation data loader, performs a forward
    pass with the model, and compares predicted PV location to "true" (gen-
    level) PV locations to compute accuracy. It then computes the average
    validation loss.

    TODO: Consider making this method more general by using terms like
    'data_loader' instead of 'val_loader', etc.
    
    Args:
        model: The neural network model to be evaluated.
        val_loader: DataLoader for the validation (evaluation) dataset.
        loss_function: The loss function used to compute validation loss.
        device: The device (e.g., 'cpu' or 'cuda') to perform computations on.
    """
    # Initialize a progress bar for validation
    pbar = NestedProgressBar(
        total_epochs=1,
        total_batches=len(val_loader),
        mode='eval',
    )
    
    model.eval()  # Set model to evaluation mode
    n_correct, n_total = 0, 0
    total_val_loss = 0

    with torch.no_grad():  # Disable gradient calculation for validation
        for batch_idx, (inputs, labels) in enumerate(val_loader):
            # Update progress bar for batch
            pbar.update_batch(batch_idx + 1)

            # Move input and labels to device
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)

            # Perform forward pass
            outputs = model(inputs)

            # Calculate validation loss for batch
            val_loss = loss_function(outputs, labels)
            # Accumulate validation loss
            total_val_loss += val_loss.data.item()

            # TODO: Get predicted PV locations
            # _, predicted = outputs.max(1)  --- IGNORE ---
            # TODO: Count correct predictions
            # n_correct += (predicted == labels).sum().item()  --- IGNORE ---
            # Count total samples in batch
            n_total = len(labels)  # Might be labels.size(0) if labels is a tensor
            
    # Close progress bar and display completion message
    pbar.close(last_message="Validation complete.")

    # Calculate accuracy
    accuracy = n_correct / n_total
    # Calculate average validation loss
    avg_val_loss = total_val_loss / len(val_loader)
    print(f"Validation loss (avg.): {avg_val_loss:.3f} | " + 
          f"Validation accuracy: {accuracy*100:.2f}%")
    # Return accuracy and total validation loss for the epoch
    return accuracy, total_val_loss

In [ ]:
# Train and evaluate the model for a number of epochs
def train_and_evaluate(model, loss_function, optimizer, train_dataloader, 
                       val_dataloader, device, n_epochs, scheduler=None):
    
    # Initialize progress bar for training
    pbar = NestedProgressBar(
        total_epochs=n_epochs,
        total_batches=len(train_dataloader),
        epoch_message_frequency=5,  # Placeholder value
        mode='train',
    )

    train_losses, val_losses, val_accuracies = [], [], []
    for epoch in range(n_epochs):
        # Update epoch-level progress bar
        pbar.update_epoch(epoch + 1)

        # Perform one epoch of training
        trained_model, train_loss = train_epoch(model, 
                                                loss_function, 
                                                optimizer,
                                                train_dataloader, 
                                                device, 
                                                pbar)
        
        # Evaluate model on validation set.
        val_accuracy, val_loss = evaluate(model, 
                                          val_dataloader, 
                                          loss_function,
                                          device)

        # Log training loss and validation metrics for current epoch
        pbar.maybe_log_epoch(epoch+1, 
                             message=f"Epoch {epoch+1} - Train loss:" + 
                             f"{train_loss:.4f} | Val loss: {val_loss:.4f} | " + 
                             f"Val accuracy: {val_accuracy*100:.2f}%")

        # Call step scheduler if there is one present
        if scheduler is not None: 
            # Step with validation loss for ReduceLROnPlateau
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)  
            # No metric needed for other schedulers
            else:
                scheduler.step()

        # Store metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)
    
    # Close progress bar and print final message upon completion
    pbar.close(message="Training complete.\n")

    metrics = [train_losses, val_losses, val_accuracies]
    # Return trained model and metrics for plotting
    return trained_model, metrics


## Define Optuna objective function

In [ ]:
def objective(trial, device):
    """
    Defines the objective function for hyperparameter optimization using Optuna.

    For each trial, this function samples a set of hyperparameters,
    constructs a model, trains it for a fixed number of epochs, evaluates
    its performance on a validation set, and returns the accuracy. Optuna
    uses the returned accuracy to guide its search for the best
    hyperparameter combination.

    Args:
        trial: An Optuna `Trial` object, used to sample hyperparameters.
        device: The device ('cpu' or 'cuda') for model training and evaluation.

    Returns:
        The validation accuracy of the trained model as a float.
    """
    # Hyperparameters for feature extractor using Optuna trial
    # TODO: method argument for values to search over for Optuna
    n_layers = trial.suggest_int("n_layers", 3, 8)
    n_filters = [trial.suggest_int(f"n_filters_{i}", 16, 64) for i in range(n_layers)]
    kernel_sizes = [trial.suggest_categorical(f"kernel_size_{i}", [3, 5, 7, 9]) for i in range(n_layers)]

    # Hyperparameters for classifier
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)

    # Instantiate model with hyperparameters
    model = FlexibleCNN(n_layers, n_filters, kernel_sizes, dropout_rate)

    # Pass dummy input through model to ensure all parameters are instantiated
    # before optimizer is defined
    dummy_input = torch.randn(1, 1, 4000).to(device)
    model(dummy_input)

    # Define other hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    # Define number of epochs
    n_epochs = 10  # TODO: method argument

    # TODO: Loss function
    loss_function = None

    # Optimizer
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs,
                                                     eta_min=1e-6)

    # Create data loaders
    train_dataloader, val_dataloader = None

    # Train model
    _, metrics = train_and_evaluate(
        model=model,
        loss_function=loss_function,
        optimizer=optimizer,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        device=device,
        n_epochs=n_epochs,
        scheduler=scheduler
    )

    # Return final validation accuracy for Optuna to optimize
    return metrics[2][-1]

## Create Optuna study

In [ ]:
# Goal is to maximize 
study = optuna.create_study(direction="maximize")

# Star optimization process (this will take a long time...)
n_trials = 100
study.optimize(lambda trial: objective(trial, device), n_trials=n_trials)

### Analyze results

In [ ]:
# Extract dataframe with results
df = study.trials_dataframe()
df

In [ ]:
# Extract and print best trial
best_trial = study.best_trial

print("Best trial:")
print(f"  Value: {best_trial.value}")
print("  Params: ")
pprint(best_trial.params)

In [ ]:
# Visualize optimization history
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title('Optimization History')
plt.show()

# Importance of hyperparameters
optuna.visualization.matplotlib.plot_param_importances(study)
plt.show()

ax = optuna.visualization.matplotlib.plot_parallel_coordinate(
    study, params=['n_layers', 'n_filters_0', 'kernel_size_0', 'dropout_rate', 'learning_rate', 'batch_size']
)
fig = ax.figure
fig.set_size_inches(12, 6, forward=True)  # forward=True updates the canvas
fig.tight_layout()